# OpenPlaque — Plaque + Inflammation Final Visualization v2

This is the single corrected **Runtime → Run all** notebook.

It reproduces the accepted plaque and direct-PCAT endpoint, then applies the visualization corrections identified in the v1 audit:

- corrected publication-style plaque colors and legend;
- exact locked PCAT segment lengths;
- true **source-space PCAT fat-voxel HU decomposition** instead of classifying 1-mm mean values;
- source-space PCAT voxel histograms;
- retained longitudinal profile and coverage-aware heatmap;
- RCA radial profile retained but explicitly labeled exploratory geometry QC.

The voxel-level decomposition is only accepted if it reproduces the locked RCA, frozen-LAD and C6/LCX PCAT means and fat-voxel counts within tight tolerances.

All figures are displayed in the notebook and exported to a single ZIP with the CSV/JSON/NPZ/HTML outputs.

**Research boundaries:** plaque volumes are OpenPlaque best-estimate proxies, not Cleerly outputs. Direct PCAT attenuation is not proprietary Caristo FAI-Score. LM inflammation is not standardized. C6 remains an LCX-like structural research segment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Paths and cache controls — immediately after Drive mount
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/OpenPlaque")
OUT = DRIVE_ROOT / "Plaque_Inflammation_Final_Visualization_v2"

REQUIRED = [
    DRIVE_ROOT / "UCLA_Plaque_Type_Estimates" / "best_estimate_plaque_types_by_artery.csv",
    DRIVE_ROOT / "Cache" / "Secondary_3D_Vesselness_Topology_v1" / "series7_int16.npy",
    DRIVE_ROOT / "Cache" / "Secondary_3D_Vesselness_Topology_v1" / "series7_int16.json",
    DRIVE_ROOT / "RCA_Ostium_TotalSegmentator" / "aorta_series7_totalseg.nii.gz",

    DRIVE_ROOT / "PCAT_RCA_10_50" / "rca_centerline_smoothed_zyx.csv",
    DRIVE_ROOT / "PCAT_RCA_10_50" / "pcat_local_radius_profile.csv",
    DRIVE_ROOT / "RCA_Plaque_PCAT_Research_Lock_v1" / "summary.json",
    DRIVE_ROOT / "RCA_Plaque_PCAT_Research_Lock_v1" / "RCA_locked_research_plaque_PCAT_profile_10_50.csv",
    DRIVE_ROOT / "PCAT_RCA_10_50_Reproducibility_Lock" / "pcat_canonical_primary_radial.csv",

    DRIVE_ROOT / "LAD_Source_Space_PCAT_Feasibility_v1" / "LAD_PCAT_source_geometry.csv",
    DRIVE_ROOT / "LAD_Source_Space_PCAT_Feasibility_v1" / "summary.json",
    DRIVE_ROOT / "LAD_Source_Space_PCAT_Feasibility_v1" / "frozen_LAD_PCAT_longitudinal.csv",

    DRIVE_ROOT / "LCX_Structural_Source_QC_Freeze_v1" / "LCX_structural_dense_source_QC.csv",
    DRIVE_ROOT / "LCX_OM_Source_Space_Composition_PCAT_Feasibility_v1" / "summary.json",
    DRIVE_ROOT / "LCX_OM_Source_Space_Composition_PCAT_Feasibility_v1" / "C6_PCAT_longitudinal_primary.csv",
    DRIVE_ROOT / "LCX_OM_Source_Space_Composition_PCAT_Feasibility_v1" / "C7_PCAT_longitudinal_primary.csv",
]

missing = [str(p) for p in REQUIRED if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing required cached inputs:\n" + "\n".join(missing))

OUT.mkdir(parents=True, exist_ok=True)
print("Output:", OUT)
print("All required source-space and endpoint caches found.")

In [ ]:
import shutil, sys, subprocess, json
from pathlib import Path

OPENPLAQUE_PIN = "2de577ea13a1fd3a3ae5bdeac2fd2657f0291bb3"
OPENPLAQUE_BRANCH = "plaque-inflammation-best-estimates-from-main"

if Path("/content/OpenPlaque").exists():
    shutil.rmtree("/content/OpenPlaque")

!git clone -q --branch {OPENPLAQUE_BRANCH} https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout -q {OPENPLAQUE_PIN}

%pip install -q /content/OpenPlaque

for name in list(sys.modules):
    if name == "openplaque" or name.startswith("openplaque."):
        del sys.modules[name]

actual = subprocess.check_output(
    ["git","-C","/content/OpenPlaque","rev-parse","HEAD"], text=True
).strip()
print("OpenPlaque pin:", actual)
assert actual == OPENPLAQUE_PIN

In [ ]:
# Synthetic/unit tests before study-data analysis
from openplaque.plaque_inflammation_final_visualization_v2 import synthetic_self_test

print(synthetic_self_test())
!cd /content/OpenPlaque && pytest -q   tests/test_plaque_inflammation_best_estimates_v1.py   tests/test_plaque_inflammation_final_visualization_v2.py

In [ ]:
# Run the complete corrected endpoint + voxel-level PCAT visualization/export pipeline
from openplaque.plaque_inflammation_final_visualization_v2 import run

summary = run(
    drive_root=str(DRIVE_ROOT),
    output_dir=str(OUT),
)

print(json.dumps(summary, indent=2))

In [ ]:
# Review the exact endpoint and voxel-validation tables
import pandas as pd
from IPython.display import display

endpoint = OUT / "endpoint"

plaque = pd.read_csv(endpoint / "plaque_best_estimates_by_vessel.csv")
aggregate = pd.read_csv(endpoint / "major_vessel_aggregate.csv")
inflammation = pd.read_csv(endpoint / "inflammation_best_estimates_by_vessel.csv")
validation = pd.read_csv(OUT / "pcat_voxel_reconstruction_validation.csv")
bands = pd.read_csv(OUT / "pcat_voxel_hu_band_decomposition.csv")

print("Plaque best estimates")
display(plaque[[
    "vessel",
    "tpv_best_estimate_mm3",
    "tpv_strict_or_known_lower_mm3",
    "tpv_candidate_envelope_upper_mm3",
    "ncpv_best_estimate_mm3",
    "lap_best_estimate_mm3",
    "calcified_plaque_volume_mm3",
    "confirm2_tpv_stage",
    "absolute_volume_confidence",
]])

print("\nMajor-vessel aggregate")
display(aggregate)

print("\nLocked direct PCAT endpoint")
display(inflammation[[
    "vessel",
    "pcat_mean_hu_best_estimate",
    "pcat_segment_length_mm",
    "segment_coverage_fraction",
    "caristo_comparability",
    "fai_score",
    "confidence",
]])

print("\nVoxel-level reconstruction validation against locked endpoints")
display(validation)

if not validation["voxel_distribution_validation_pass"].all():
    raise RuntimeError("Voxel-level PCAT reconstruction did not reproduce the locked endpoints.")

print("\nTrue source-space PCAT voxel HU bands")
display(bands)

In [ ]:
# Display every corrected figure inline
from IPython.display import Image, display

FIGURES = [
    "01_plaque_composition_corrected.png",
    "02_inflammation_mean_pcat_corrected.png",
    "03_inflammation_voxel_hu_decomposition.png",
    "04_inflammation_voxel_hu_histograms.png",
    "05_inflammation_longitudinal_profiles.png",
    "06_inflammation_longitudinal_heatmap.png",
    "07_rca_radial_pcat_gradient_descriptive.png",
]

for name in FIGURES:
    p = OUT / name
    if not p.is_file():
        raise FileNotFoundError(p)
    print("\n" + name)
    display(Image(filename=str(p), width=1200))

In [ ]:
# Display the complete HTML report
from IPython.display import HTML, display

report = OUT / "OPENPLAQUE_PLAQUE_INFLAMMATION_FINAL_VISUALIZATION_V2_REPORT.html"
display(HTML(report.read_text()))

In [ ]:
# Verify final ZIP and all deliverables
ZIP_PATH = OUT / "OPENPLAQUE_PLAQUE_INFLAMMATION_FINAL_VISUALIZATION_V2_RESULTS.zip"

expected = [
    "run_state.json",
    "summary.json",
    "pcat_voxel_reconstruction_validation.csv",
    "pcat_voxel_hu_band_decomposition.csv",
    "pcat_voxel_values.npz",
    "pcat_longitudinal_profiles_used.csv",
    "rca_locked_radial_profile_used.csv",
    "01_plaque_composition_corrected.png",
    "02_inflammation_mean_pcat_corrected.png",
    "03_inflammation_voxel_hu_decomposition.png",
    "04_inflammation_voxel_hu_histograms.png",
    "05_inflammation_longitudinal_profiles.png",
    "06_inflammation_longitudinal_heatmap.png",
    "07_rca_radial_pcat_gradient_descriptive.png",
    "OPENPLAQUE_PLAQUE_INFLAMMATION_FINAL_VISUALIZATION_V2_REPORT.html",
    "OPENPLAQUE_PLAQUE_INFLAMMATION_FINAL_VISUALIZATION_V2_RESULTS.zip",
]

missing = [x for x in expected if not (OUT / x).exists()]
if missing:
    raise RuntimeError("Missing outputs: " + str(missing))

state = json.loads((OUT / "run_state.json").read_text())
if state.get("status") != "COMPLETE":
    raise RuntimeError("Run state not COMPLETE: " + str(state))

print("COMPLETE")
print("ZIP:", ZIP_PATH)
print("ZIP size (MB):", ZIP_PATH.stat().st_size / 1e6)

print("\nTop-level output files:")
for p in sorted(OUT.iterdir()):
    if p.is_file():
        print(" ", p.name)